# 25_cited_text: text for cited works outside the pulled set (pilot test, 2026-11-12)

Thin driver: logic lives in `citecurv.data.cited` and `citecurv.embed.specter2`. Runtime: **GPU** (a T4 is enough),
runtime version 26.07 (Python 3.12).

Why: `10_data` stores a record only for the works that match the slice filter. The works they cite are ids
without title, abstract or date; `20_embed` gives them no vector, and the TGN sees every such work with no
earlier citation as the same node (PROGRESS.md, S1.7). This notebook fetches title and abstract for those ids
from OpenAlex, encodes them with SPECTER2 and writes a second embeddings file. `40_tgn` reads it through
`EXTRA_EMB_PATH`.

Reads from `MyDrive/citecurv/<RUN>/`: `events.parquet`, `works.parquet`, `event_labels.parquet` (optional).
Writes there: `cold_start_stats.json`, `cited_works.parquet` (+ `cited_works_stats.json`),
`embeddings_cited.npz`. Needs the Colab secret `OPENALEX_API_KEY`. Both long steps resume after a disconnect:
rerun all cells.

Cost: one OpenAlex list call per 100 ids (10 credits each on a free key, 100,000 credits a day).

In [ ]:
# ---- Python version gate -----------------------------------------------------------------------------
import platform
import sys

print("python  ", sys.version)
if sys.version_info[:2] != (3, 12):
    raise RuntimeError(f"This notebook needs Python 3.12, found {platform.python_version()}. In Colab: Runtime > "
                       "Change runtime type > Runtime version 26.07 (Python 3.12), Save, then Run all again.")

In [ ]:
# ---- install citecurv with the data-stage and embed-stage pins (configs/pins.md) -----------------------
import subprocess

REPO = "https://github.com/Natboa/Detecting-Network-Citation-Anomalies-Using-Graph-Curvature"
REPO_TAG = "main"   # pilot test; set a release tag for a run that goes into the results
subprocess.run(["rm", "-rf", "/content/repo"], check=True)   # a rerun clones again
subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_TAG, REPO, "/content/repo"], check=True)
print("commit", subprocess.run(["git", "-C", "/content/repo", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())
%pip install -q /content/repo "pyalex==0.21" pyarrow requests pyyaml "adapters==1.3.0" "transformers==4.57.6"

import importlib.metadata as md

VERSIONS = {"python": sys.version.split()[0]}
for pkg in ("torch", "transformers", "adapters", "citecurv", "pyalex", "numpy", "pandas"):
    VERSIONS[pkg] = md.version(pkg)
print(VERSIONS)
assert VERSIONS["adapters"] == "1.3.0" and VERSIONS["transformers"] == "4.57.6", (
    "embed pins not active: restart the runtime (Runtime > Restart session) and rerun this cell")

In [ ]:
# ---- Drive, API key (Colab Secrets), settings ------------------------------------------------------------
import json
import os
from pathlib import Path

from google.colab import drive, userdata

drive.mount("/content/drive")
os.environ["OPENALEX_API_KEY"] = userdata.get("OPENALEX_API_KEY")   # never print or save it

from citecurv.config import load_config

cfg = load_config("/content/repo/configs/default.yaml")
RUN = "pilot"                                   # same value as in 10_data
RUN_DIR = Path("/content/drive/MyDrive") / "citecurv" / RUN
EVENTS = RUN_DIR / "events.parquet"
WORKS = RUN_DIR / "works.parquet"
EVENT_LABELS = RUN_DIR / "event_labels.parquet"
CITED_WORKS = RUN_DIR / "cited_works.parquet"
OUT = RUN_DIR / "embeddings_cited.npz"
CHUNK_DIR = RUN_DIR / "embed_chunks_cited"       # resume parts of the encoding
COLD_START_K = cfg["scoring"]["cold_start_k"]    # 3 (provisional, L19)
BATCH_SIZE = 32
CHUNK_SIZE = 20_000
assert EVENTS.exists() and WORKS.exists(), f"run 10_data with RUN = {RUN!r} first"
print(RUN_DIR, "| cold_start_k", COLD_START_K)

In [ ]:
# ---- which cited works have no record, and how many events the cold-start rule leaves out ------------------
import pandas as pd

from citecurv.data.cited import cited_only_ids, cold_start_summary
from citecurv.data.events import read_events

events = read_events(EVENTS)
works = pd.read_parquet(WORKS, columns=["work_id"])
ids = cited_only_ids(events, works)
print(f"{len(events):,} events | {events['dst_id'].nunique():,} cited works | {len(ids):,} of them without a record")

event_labels = pd.read_parquet(EVENT_LABELS) if EVENT_LABELS.exists() else None
cold = cold_start_summary(events, k=COLD_START_K, event_labels=event_labels)
(RUN_DIR / "cold_start_stats.json").write_text(json.dumps(cold, indent=1))
print(cold)

In [ ]:
# ---- OpenAlex credits left today, and what this pull needs ---------------------------------------------------
import math

import requests

r = requests.get("https://api.openalex.org/rate-limit", params={"api_key": os.environ["OPENALEX_API_KEY"]}, timeout=30)
print(r.status_code, (r.json() if r.ok else {}).get("rate_limit"))
print("calls needed:", math.ceil(len(ids) / 100), "(10 credits each on a free key)")

In [ ]:
# ---- fetch title, abstract and date by id (resumable: finished parts are kept in cited_works_parts/) --------
import time

from citecurv.data.cited import pull_works_by_ids, resolve_id_filter
from citecurv.data.openalex import BudgetExhausted

# The docs show the many-values form for DOIs only, so the id attribute is found by a probe: two pulled
# works (known to exist) must come back. One list call per name tried.
ID_FILTER = resolve_id_filter(list(works["work_id"].head(2)), cfg)
print("id filter:", ID_FILTER)
t0 = time.perf_counter()
try:
    pull_stats = pull_works_by_ids(ids, CITED_WORKS, config=cfg, id_filter=ID_FILTER, log=print)
except BudgetExhausted as e:
    print(e)   # reset time (UTC); finished parts are saved, rerun this notebook after the reset
    raise
pull_stats["minutes"] = round((time.perf_counter() - t0) / 60, 2)
print(pull_stats)

In [ ]:
# ---- SPECTER2 on the fetched works that have any text (resumable) ----------------------------------------------
import torch

from citecurv.embed.specter2 import encode_resumable, load_embeddings_npz, load_specter2, write_embeddings_npz

assert torch.cuda.is_available(), "Switch the runtime to a GPU (Runtime > Change runtime type)"
GPU = torch.cuda.get_device_name(0)
cited = pd.read_parquet(CITED_WORKS)
has_title = cited["title"].fillna("").str.strip().ne("")
has_abstract = cited["abstract"].fillna("").str.strip().ne("")
# a work with neither title nor abstract would get the vector of an empty text; it stays without a vector instead
cited = cited[has_title | has_abstract].reset_index(drop=True)
print(GPU, f"| {len(cited):,} works to encode; {has_abstract.sum():,} with abstract; "
      f"{(~(has_title | has_abstract)).sum():,} without any text (left out)")

tokenizer, model = load_specter2("cuda")
t0 = time.perf_counter()
vectors, n_new = encode_resumable(cited, CHUNK_DIR, chunk_size=CHUNK_SIZE, batch_size=BATCH_SIZE,
                                  model=model, tokenizer=tokenizer, device="cuda")
elapsed = time.perf_counter() - t0
write_embeddings_npz(OUT, cited["work_id"].to_numpy(), vectors)
out_ids, out_vecs = load_embeddings_npz(OUT)
assert len(out_ids) == len(cited) and out_vecs.shape == (len(cited), 768) and out_vecs.dtype == "float32"
print(OUT, out_vecs.shape)

In [ ]:
# ---- record for PROGRESS.md (add the Colab unit burn by hand) ----------------------------------------------------
import datetime

record = {
    "task": "S1.7 follow-up: text for cited works",
    "date": datetime.date.today().isoformat(),
    "run": RUN,
    "gpu": GPU,
    "cold_start": cold,
    "pull": pull_stats,
    "n_encoded": int(len(cited)),
    "n_encoded_this_session": int(n_new),
    "encode_minutes_this_session": round(elapsed / 60, 2),
    "docs_per_min": None if n_new == 0 else round(n_new / (elapsed / 60)),
    "versions": VERSIONS,
}
(RUN_DIR / "cited_text_record.json").write_text(json.dumps(record, indent=1))
print(json.dumps(record, indent=1))